In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# =========================
# 1. LIST PAGE URL
# =========================
BASE_LIST_URL = "https://immovlan.be/en/real-estate?transactiontypes=for-sale,in-public-sale&page={}"
# base = "https://immovlan.be/en/real-estate"

# params = {
#     "transactiontypes": "for-sale",
#     "propertytypes": "house,apartment",
#     "propertysubtypes": "residence,villa,mixed-building,master-house,bungalow,cottage,chalet,mansion,apartment,ground-floor,penthouse,duplex,studio,loft,triplex",
#     "islifeannuity": "no",
#     "noindex": "1"
# }

# url = base + "?" + "&".join([f"{k}={v}" for k, v in params.items()])

# print(url)

def get_listing_links(page):
    url = BASE_LIST_URL.format(page)
    r = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(r.text, "lxml")
    links = []

    for a in soup.select("a[href]"):
        href = a.get("href")

        if href and "/en/detail/" in href:
            # full_url = "https://immovlan.be" + href
            links.append(href)

    print("page:", page, "links:", len(links))
    return list(set(links))


# =========================
# 2. PROPERTY DETAIL SCRAPER
# =========================
def parse_property(url):
    r = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(r.text, "lxml")
    print(soup)

    content = soup.find("div", id="main_content")
    name = content.find("span", class_="detail__header_title_main")

    h1 = soup.find("h1", id="firstHeading")
    span = h1.find("span", class_="mw-page-title-main") if h1 else None
    # page_title = span.get_text(strip=True) if span else ""

    # def get_text(selector):
    #     el = soup.select_one(selector)
    #     return el.text.strip() if el else None

    # data = {
    #     "url": url,
    #     "id": url.split("/")[-1],
    #     "title": get_text("h1"),
    #     "price": get_text(".price"),
    #     "location": get_text(".locality"),
    #     "bedrooms": get_text("[data-testid='bedrooms']"),
    #     "surface": get_text("[data-testid='surface']")
    # }

    return data


# =========================
# 3. COLLECT ALL LISTING LINKS
# =========================
all_links = []

for page in range(1, 3):  # increase range if needed to reach 10k records
    try:
        links = get_listing_links(page)
        all_links.extend(links)
        print(f"Page {page} -> total links collected: {len(all_links)}")
        time.sleep(1)
    except:
        continue

# remove duplicate URLs
print("all_links")
print(all_links)
all_links = list(set(all_links))
print("remove duplicate URLs all_links")
print(all_links)


# =========================
# 4. SCRAPE PROPERTY DETAILS
# =========================
dataset = []

# for url in tqdm(all_links[:10]):  # limit to 10k records
for url in all_links:  # limit to 10k records
    try:
        data = parse_property(url)
        dataset.append(data)
        time.sleep(0.5)  # prevent blocking
    except:
        continue

print("dataset")
print(dataset)

# =========================
# 5. SAVE DATASET
# =========================
# df = pd.DataFrame(dataset)
# df.to_csv("belgium_properties.csv", index=False)

print("DONE -> dataset saved with 10k+ properties")

page: 1 links: 72
Page 1 -> total links collected: 18
page: 2 links: 80
Page 2 -> total links collected: 38
all_links
['https://immovlan.be/en/detail/land/for-sale/6110/montigny-le-tilleul/vbe31885', 'https://immovlan.be/en/detail/apartment/for-sale/8400/oostende/rbw13583', 'https://immovlan.be/en/detail/residence/for-sale/8930/rekkem/rbw13670', 'https://immovlan.be/en/detail/garage/for-sale/6030/marchienne-au-pont/vbe31975', 'https://immovlan.be/en/detail/apartment/for-sale/6880/bertrix/vbe31926', 'https://immovlan.be/en/detail/business-surface/for-sale/2500/lier/rbw13366', 'https://immovlan.be/en/detail/office-space/for-sale/2240/zandhoven/vbe32003', 'https://immovlan.be/en/detail/residence/for-sale/6280/gerpinnes/vwd17735', 'https://immovlan.be/en/detail/residence/for-sale/1820/steenokkerzeel/vbe32011', 'https://immovlan.be/en/detail/residence/for-sale/2840/rumst/rbw13365', 'https://immovlan.be/en/detail/land/for-sale/5060/arsimont/vbe31782', 'https://immovlan.be/en/detail/mixed-bui